# Interpolate data onto an even time step
Author: Brian Green (briangre@stanford.edu)

In [1]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import netCDF4 as nc
from scipy import interpolate

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    tqdm.pandas()

%matplotlib inline

INTERP_DT = '30S' # the time step to interpolate the data to
INTERP_DT_NP = np.timedelta64(30,'s') # the same as above, but in a format that numpy can use

In [2]:
# Load in the data
flights = pd.read_feather('Strateole2_with_ERA5_3_depressurizations.feather')
flights

,flight_id,time,latitude,longitude,altitude,u,v,pressure,T_ERA5,N2_ERA5,superpressure,segment_id
0,C0_01_STR1,2019-11-12 22:25:07.005612,-4.834385,55.593623,20306.0,NaN,NaN,53.838274,203.456487,0.000691,8.270000,0
1,C0_01_STR1,2019-11-12 22:25:37.076783,-4.836663,55.593735,20297.0,0.361103,-8.387708,53.902477,203.428319,0.000691,8.270000,0
2,C0_01_STR1,2019-11-12 22:26:07.026208,-4.838898,55.593818,20297.0,0.256162,-8.362772,53.893533,203.432243,0.000691,8.200000,0
3,C0_01_STR1,2019-11-12 22:26:37.098505,-4.841163,55.593873,20305.0,0.200608,-8.576493,53.804733,203.471199,0.000691,8.200000,0
4,C0_01_STR1,2019-11-12 22:27:07.047890,-4.843513,55.593927,20306.0,0.188386,-8.961514,53.773176,203.485038,0.000691,8.770000,0
...,...,...,...,...,...,...,...,...,...,...,...,...
3664718,C1_17_TTL3,2021-12-16 12:31:57.000000,-2.211255,125.868032,18601.4,-3.421231,2.695646,71.513419,191.921589,0.000809,10.187500,74
3664719,C1_17_TTL3,2021-12-16 12:32:27.000000,-2.210570,125.867093,18591.3,-3.461483,2.463263,71.650113,191.897698,0.000808,9.933594,74
3664720,C1_17_TTL3,2021-12-16 12:32:57.000000,-2.209930,125.866168,18597.9,-3.272620,2.364113,71.578122,191.910271,0.000809,9.821289,74
3664721,C1_17_TTL3,2021-12-16 12:33:27.000000,-2.209298,125.865332,18611.9,-2.845354,2.463263,71.404752,191.940639,0.000809,9.756836,74


In [3]:
def interp_segment(segment):
    
    # Make the data numpy arrays
    time = np.array(segment.time)
    latitude = np.array(segment.latitude)
    longitude = np.array(segment.longitude)
    wind_u = np.array(segment.u)
    wind_v = np.array(segment.v)
    altitude = np.array(segment.altitude)
    pressure = np.array(segment.pressure)
    T_ERA5 = np.array(segment.T_ERA5)
    N2_ERA5 = np.array(segment.N2_ERA5)
    superpressure = np.array(segment.superpressure)
    
    # Set the time range to prevent extrapolation if there are NaNs in the wind, altitude, or pressure data
    temp = wind_u + wind_v + altitude + pressure
    temp_time = time[np.isnan(temp) == 0]
    
    # Create the new time series and the dataframe
    # Set the start time so that it's the first multiple of INTERP_DT_NP after the start
    # of the segment. This ensures all the segments are on the same time grid.
    ref_time = np.datetime64('2000-01-01T00:00:00.000000000')
    start_time = np.ceil((time[0] - ref_time)/INTERP_DT_NP)
    start_time = ref_time + INTERP_DT_NP*start_time
    new_time = pd.date_range(start_time, temp_time[-1], freq=INTERP_DT)
    interpd = pd.DataFrame(columns=['flight_id','time','latitude','longitude','altitude',
                                    'pressure','wind_u','wind_v','T_ERA5','N2_ERA5',
                                    'superpressure','segment_id'])
    interpd.time = new_time
    interpd.flight_id = segment.flight_id.iloc[0]
    interpd.segment_id = segment.segment_id.iloc[0]
    
    # Times must be converted to a float before fields can be interpolated
    time_float = np.array((time - time[0])/pd.Timedelta(1, 'minutes'))
    new_time_float = np.array((new_time - time[0])/pd.Timedelta(1, 'minutes'))
    N_temp = np.size(new_time_float)
    
    # There can be NaNs in some fields. Delete these and make new time series for interpolation
    time_float_lat = time_float[np.isnan(latitude) == 0]
    time_float_lon = time_float[np.isnan(longitude) == 0]
    time_float_alt = time_float[np.isnan(altitude) == 0]
    time_float_u = time_float[np.isnan(wind_u) == 0]
    time_float_v = time_float[np.isnan(wind_v) == 0]
    time_float_p = time_float[np.isnan(pressure) == 0]
    time_float_T = time_float[np.isnan(T_ERA5) == 0]
    time_float_N2 = time_float[np.isnan(N2_ERA5) == 0]
    time_float_SP = time_float[np.isnan(superpressure) == 0]
    latitude = latitude[np.isnan(latitude) == 0]
    longitude = longitude[np.isnan(longitude) == 0]
    altitude = altitude[np.isnan(altitude) == 0]
    wind_u = wind_u[np.isnan(wind_u) == 0]
    wind_v = wind_v[np.isnan(wind_v) == 0]
    pressure = pressure[np.isnan(pressure) == 0]
    T_ERA5 = T_ERA5[np.isnan(T_ERA5) == 0]
    N2_ERA5 = N2_ERA5[np.isnan(N2_ERA5) == 0]
    superpressure = superpressure[np.isnan(superpressure) == 0]
    # For dealing with the date line:
    temp_longitude = longitude.copy()
    temp_longitude[temp_longitude < 0] = temp_longitude[temp_longitude < 0] + 360
    
    # Do the interpolation
    f1 = interpolate.PchipInterpolator(time_float_lat, latitude)
    f2 = interpolate.PchipInterpolator(time_float_lon, longitude)
    f2b = interpolate.PchipInterpolator(time_float_lon, temp_longitude)
    f3 = interpolate.PchipInterpolator(time_float_alt, altitude)
    f4 = interpolate.PchipInterpolator(time_float_p, pressure)
    f5 = interpolate.PchipInterpolator(time_float_u, wind_u)
    f6 = interpolate.PchipInterpolator(time_float_v, wind_v)
    if T_ERA5.size > 0: # one flight doesn't have any ERA5 data
        f7 = interpolate.PchipInterpolator(time_float_T, T_ERA5)
        f8 = interpolate.PchipInterpolator(time_float_N2, N2_ERA5)
    f9 = interpolate.PchipInterpolator(time_float_SP, superpressure)
    interpd.latitude = f1(new_time_float)
    longitude1 = f2(new_time_float)
    longitude2 = f2b(new_time_float)
    interpd.altitude = f3(new_time_float)
    interpd.pressure = f4(new_time_float)
    interpd.wind_u = f5(new_time_float)
    interpd.wind_v = f6(new_time_float)
    if T_ERA5.size > 0: # one flight doesn't have any ERA5 data
        interpd.T_ERA5 = f7(new_time_float)
        interpd.N2_ERA5 = f8(new_time_float)
    else:
        interpd.T_ERA5 = np.nan
        interpd.N2_ERA5 = np.nan
    interpd.superpressure = f9(new_time_float)
    
    ## Deal with balloons crossing the date line
    # Identify intervals where the balloon crossed the date line
    dlon = np.array(segment.longitude)
    dlon = np.diff(dlon)
    indices = 1 + np.arange(np.size(dlon))
    indices_new = np.arange(np.size(new_time_float))
    if np.nanmax(np.abs(dlon)) < 180:
        interpd.longitude = longitude1
    else:
        ind_crossings = indices[np.abs(dlon) > 180]
        n_crossings = np.size(ind_crossings)
        for i in np.arange(n_crossings):
            ind_start = ind_crossings[i] - 1
            ind_stop = ind_crossings[i]
            time_float_start = time_float[ind_start]
            time_float_stop = time_float[ind_stop]
            # Find new_time_float indices in this time range
            temp_time = new_time_float.copy()
            ind_new_crossing = indices_new[temp_time > time_float_start]
            temp_time = temp_time[temp_time > time_float_start]
            ind_new_crossing = ind_new_crossing[temp_time < time_float_stop]
            # Replace longitude1 with longitude2 for those indices
            longitude1[ind_new_crossing] = longitude2[ind_new_crossing]
        # Make sure the longitude range is -180 to 180
        longitude1[longitude1 > 180] = longitude1[longitude1 > 180] - 360
        interpd.longitude = longitude1
    
    return interpd

flights_interpolated = flights.groupby('segment_id').progress_apply(interp_segment).reset_index(drop=True)

  0%|          | 0/57 [00:00<?, ?it/s]

In [4]:
# Save the data
flights_interpolated.to_feather('Strateole2_with_ERA5_4_interpolated.feather') 
flights_interpolated

,flight_id,time,latitude,longitude,altitude,pressure,wind_u,wind_v,T_ERA5,N2_ERA5,superpressure,segment_id
0,C0_01_STR1,2019-11-12 22:26:00,-4.838373,55.593802,20297.000000,53.897010,0.275275,-8.363094,203.430718,0.000691,8.209750,0
1,C0_01_STR1,2019-11-12 22:28:00,-4.847736,55.593991,20290.601597,53.883324,0.119288,-8.672393,203.436723,0.000691,8.083821,0
2,C0_01_STR1,2019-11-12 22:30:00,-4.857348,55.594430,20308.949840,53.695757,0.583395,-9.098239,203.486325,0.000694,8.180535,0
3,C0_01_STR1,2019-11-12 22:32:00,-4.866458,55.594784,20287.363769,53.917584,0.284297,-8.289193,203.373900,0.000695,7.975163,0
4,C0_01_STR1,2019-11-12 22:34:00,-4.875593,55.595684,20320.073996,53.663059,1.274234,-8.547122,203.491664,0.000694,8.191583,0
...,...,...,...,...,...,...,...,...,...,...,...,...
921373,C1_17_TTL3,2021-12-16 12:24:00,-2.223677,125.880593,18649.352150,70.901548,-3.111353,2.058237,191.862282,0.000809,11.958291,74
921374,C1_17_TTL3,2021-12-16 12:26:00,-2.220984,125.877360,18628.157799,71.157458,-2.846830,2.898892,191.818788,0.000809,11.790392,74
921375,C1_17_TTL3,2021-12-16 12:28:00,-2.217645,125.874561,18636.931769,71.031784,-2.633165,3.173405,191.840114,0.000809,11.506331,74
921376,C1_17_TTL3,2021-12-16 12:30:00,-2.214474,125.871257,18635.161510,71.081333,-3.026773,2.973111,191.839663,0.000809,11.081188,74
